<a href="https://colab.research.google.com/github/dswhaley/UFC-Fight-Prediction/blob/main/UFC_Project_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UFC Project 2
## Adding the Data Cleaning from Project 1

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os


pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 500)
pd.set_option('display.expand_frame_repr', False)

path = kagglehub.dataset_download("mdabbert/ultimate-ufc-dataset")



In [ ]:
df = pd.read_csv(path + '/ufc-master.csv')
df = df.drop(columns=['RedDecOdds', 'BlueDecOdds', 'RSubOdds', 'BSubOdds', 'RKOOdds', 'BKOOdds', 'FinishDetails','EmptyArena'])


In [ ]:
red_rank_cols = [
    'RWFlyweightRank', 'RWFeatherweightRank', 'RWStrawweightRank', 'RWBantamweightRank',
    'RHeavyweightRank', 'RLightHeavyweightRank', 'RMiddleweightRank', 'RWelterweightRank',
    'RLightweightRank', 'RFeatherweightRank', 'RBantamweightRank', 'RFlyweightRank', 'RMatchWCRank'
]

blue_rank_cols = [
    'BWFlyweightRank', 'BWFeatherweightRank', 'BWStrawweightRank', 'BWBantamweightRank',
    'BHeavyweightRank', 'BLightHeavyweightRank', 'BMiddleweightRank', 'BWelterweightRank',
    'BLightweightRank', 'BFeatherweightRank', 'BBantamweightRank', 'BFlyweightRank', 'BMatchWCRank'
]
def createdict(cols, row):
  wcRanks = {}
  for col in cols:
    wcRanks[col] = row[col]
  return wcRanks

def check_if_ranked(color, weightClass, wcranks):
  for key, value in wcranks.items():
    if pd.notna(value) and key != f'{color}{weightClass}Rank':
      return True
  return False


unranked_fighters_ranked_in_other_weightclass_fighting_ranked_opponents = 0
uranked_fighters_unranked_in_other_weightclassess_fighting_ranked_opponents = 0
case1UnrankedWins= 0
case1RankedWins = 0
case2UnrankedWins = 0
case2RankedWins = 0




for index, row in df.iterrows():
  weightClass = row['WeightClass']
  weightClass = weightClass.replace(" ", "").replace("'", "")

  if weightClass == 'CatchWeight' or 'Women' in weightClass: # There are no ranks in the df for women or catchweight fights
    continue
  red_dict = createdict(red_rank_cols, row)
  blue_dict = createdict(blue_rank_cols, row)


  if(pd.isna(row[f'B{weightClass}Rank'])) and pd.notna(row[f'R{weightClass}Rank']):
    if(check_if_ranked('B', weightClass, blue_dict)):
      unranked_fighters_ranked_in_other_weightclass_fighting_ranked_opponents += 1
      if(row['Winner'] == 'Red'):
        case1RankedWins += 1
      else:
        case1UnrankedWins += 1
    else:
      uranked_fighters_unranked_in_other_weightclassess_fighting_ranked_opponents += 1
      if(row['Winner'] == 'Red'):
        case2RankedWins += 1
      else:
        case2UnrankedWins += 1
  elif pd.notna(row[f'B{weightClass}Rank']) and pd.isna(row[f'R{weightClass}Rank']):
    if(check_if_ranked('R', weightClass, red_dict)):
      unranked_fighters_ranked_in_other_weightclass_fighting_ranked_opponents += 1
      if(row['Winner'] == 'Red'):
        case1UnrankedWins += 1
      else:
        case1RankedWins += 1
    else:
      uranked_fighters_unranked_in_other_weightclassess_fighting_ranked_opponents += 1
      if(row['Winner'] == 'Red'):
        case2UnrankedWins += 1
      else:
        case2RankedWins += 1

print(f'There are {unranked_fighters_ranked_in_other_weightclass_fighting_ranked_opponents} fighters fighting unranked in fights when they are ranked in other weight classes \n The unranked fighter has won {case1UnrankedWins} times. \nThis means that when a fighter ranked in another weightclass fights a fighter unranked in another weightclass they win {(case1UnrankedWins/unranked_fighters_ranked_in_other_weightclass_fighting_ranked_opponents)* 100}% of the time ')

print('\n\n\n')
print(f'There are {uranked_fighters_unranked_in_other_weightclassess_fighting_ranked_opponents} fighters fighting unranked in fights when they are unranked in other weight classes \nIn those cases, the unranked fighter won {case2UnrankedWins} times. \nThis means that when a fighter unranked fights a fighter unranked in another weightclass they win {(case2UnrankedWins/uranked_fighters_unranked_in_other_weightclassess_fighting_ranked_opponents) * 100}% of the time ')

In [ ]:
def invert_rank(rank):
    if pd.isna(rank):
        return 0
    else:
        return 16 - rank

all_rank_cols = red_rank_cols + blue_rank_cols
for col in all_rank_cols:
    df[col] = df[col].apply(invert_rank)


In [ ]:

# I do not have much to any experience with pandas, so I used ChatGPT to get a better understanding for how to manipulate dfs. I didn't know if I should iterate over  the whole df. It told me to use '.apply()'. Other than that I created the functions myself besides using chatgpt like google to learn about pandas. I did not have it generate any code.






def set_wc_rank(color: str, row):
  if color == 'R':
    cols = red_rank_cols
  else:
    cols = blue_rank_cols
  wcRanks = createdict(cols, row)
  weightClass = row['WeightClass']
  weightClass = weightClass.replace(" ", "").replace("'", "")
  rank_value = wcRanks.get(f'{color}{weightClass}Rank')

  if pd.notna(rank_value):
    return wcRanks.get(f'{color}{weightClass}Rank')

  return 0





red_rank_cols = [
      'RWFlyweightRank',
      'RWFeatherweightRank',
      'RWStrawweightRank',
      'RWBantamweightRank',
      'RHeavyweightRank',
      'RLightHeavyweightRank',
      'RMiddleweightRank',
      'RWelterweightRank',
      'RLightweightRank',
      'RFeatherweightRank',
      'RBantamweightRank',
      'RFlyweightRank',
      'RMatchWCRank'
]

blue_rank_cols = [
    'BWFlyweightRank',
    'BWFeatherweightRank',
    'BWStrawweightRank',
    'BWBantamweightRank',
    'BHeavyweightRank',
    'BLightHeavyweightRank',
    'BMiddleweightRank',
    'BWelterweightRank',
    'BLightweightRank',
    'BFeatherweightRank',
    'BBantamweightRank',
    'BFlyweightRank',
    'BMatchWCRank'
  ]


df['RWeightClassRank'] = df.apply(lambda row: set_wc_rank('R', row), axis=1)
df['BWeightClassRank'] = df.apply(lambda row: set_wc_rank('B', row), axis=1)


df.head()

In [ ]:
df = df.drop(columns=all_rank_cols)
df['Winner'] = df['Winner'].apply(lambda x: 1 if x == 'Red' else 0)


In [ ]:
df['BPFPRank'] = df['BPFPRank'].apply(invert_rank)
df['RPFPRank'] = df['RPFPRank'].apply(invert_rank)

df['BPFPRank'] = df['BPFPRank'].apply(lambda x: 0 if pd.isna(x) else x)
df['RPFPRank'] = df['RPFPRank'].apply(lambda x: 0 if pd.isna(x) else x)


In [ ]:
df['RBetterRank'] = df['BetterRank'].apply(lambda x: 1 if x == 'Red' else 0)
df['BBetterRank'] = df['BetterRank'].apply(lambda x: 1 if x == 'Blue' else 0)
df = df.drop(columns=['BetterRank'])

In [ ]:
df['TitleBout'] = df['TitleBout'].apply(lambda x: 1 if x == True else 0)


In [ ]:
df['GenderMale'] = df['Gender'].apply(lambda x: 1 if x == 'MALE' else 0)
df['GenderFemale'] = df['Gender'].apply(lambda x: 1 if x == 'FEMALE' else 0)
df = df.drop(columns=['Gender'])


In [ ]:
df['RSouthpaw'] = df['RedStance'].apply(lambda x: 1 if x == 'Southpaw' else 0)
df['ROthodox'] = df['RedStance'].apply(lambda x: 1 if x == 'Orthodox' else 0)
df['BSouthpaw'] = df['BlueStance'].apply(lambda x: 1 if x == 'Southpaw' else 0)
df['BOthodox'] = df['BlueStance'].apply(lambda x: 1 if x == 'Orthodox' else 0)

df['differentStance'] = df.apply(lambda row: 1 if row['RedStance'] != row['BlueStance'] else 0, axis=1)

df = df.drop(columns=['RedStance', 'BlueStance'])

In [ ]:
df['UDec'] = df['Finish'].apply(lambda x: 1 if x == 'U-DEC' else 0)
df['SDec'] = df['Finish'].apply(lambda x: 1 if x == 'S-DEC' else 0)
df['Sub'] = df['Finish'].apply(lambda x: 1 if x == 'SUB' else 0)
df['KO/TKO'] = df['Finish'].apply(lambda x: 1 if x == 'KO/TKO' else 0)

df = df.drop(columns=['Finish'])

In [ ]:
weight_mapping = {
    "Women's Strawweight": 1,
    "Women's Flyweight": 2,
    "Flyweight": 3,
    "Bantamweight": 4,
    "Women's Bantamweight": 5,
    "Featherweight": 6,
    "Women's Featherweight": 7,
    "Lightweight": 8,
    "Welterweight": 9,
    "Middleweight": 10,
    "Light Heavyweight": 11,
    "Heavyweight": 12,
    "Catch Weight": 13
}

# Apply mapping to create numerical column
df['weight_class_num'] = df['WeightClass'].map(weight_mapping)

In [ ]:

def koed_last_fight(fighter, row):
  fighter_fights = df[
          (df['RedFighter'] == fighter) |
          (df['BlueFighter'] == fighter)
      ]
  last_fight = fighter_fights.iloc[0]
  if(last_fight['RedFighter'] == fighter and last_fight['Winner'] == 0 and last_fight['KO/TKO'] == 1):
    return 1
  elif(last_fight['BlueFighter'] == fighter and last_fight['Winner'] == 1 and last_fight['KO/TKO'] == 1):
    return 1

  return 0

df['BLastFightWasKOed'] = df.apply(lambda row: koed_last_fight(row['BlueFighter'], row), axis=1)
df['RLastFightWasKOed'] = df.apply(lambda row: koed_last_fight(row['RedFighter'], row), axis=1)


In [ ]:
fighter_fights = df[
          (df['RedFighter'] == 'Conor McGregor') |
          (df['BlueFighter'] == 'Conor McGregor')
      ]

In [ ]:
# Combine fighters who were KOed in their last fight
fighters_koed_in_last_fight = pd.concat([
    df[df['RLastFightWasKOed'] == 1][['RedFighter', 'Winner', 'RLastFightWasKOed', 'RedAge']].assign(Color='Red'),
    df[df['BLastFightWasKOed'] == 1][['BlueFighter', 'Winner', 'BLastFightWasKOed', 'BlueAge']].assign(Color='Blue')
], ignore_index=True)

# Rename age column for consistency and ensure it's a single column
fighters_koed_in_last_fight['Age'] = fighters_koed_in_last_fight['RedAge'].fillna(fighters_koed_in_last_fight['BlueAge'])
fighters_koed_in_last_fight = fighters_koed_in_last_fight.drop(columns=['RedAge', 'BlueAge'])


# Add a column 'LostNextFight' (1 if lost, 0 if won)
def determine_loss(row):
    if row['Color'] == 'Red' and row['Winner'] == 0:
        return 1
    elif row['Color'] == 'Blue' and row['Winner'] == 1:
        return 1
    else:
        return 0

fighters_koed_in_last_fight['LostNextFight'] = fighters_koed_in_last_fight.apply(determine_loss, axis=1)

In [ ]:
grappler_metrics = {
    'AvgSubAtt': pd.concat([df['RedAvgSubAtt'], df['BlueAvgSubAtt']]).mean(),
    'WinsBySubmission': pd.concat([df['RedWinsBySubmission'], df['BlueWinsBySubmission']]).mean(),
}

wrestler_metrics = {
    'AvgTDLanded': pd.concat([df['RedAvgTDLanded'], df['BlueAvgTDLanded']]).mean(),
    'AvgTDPct': pd.concat([df['RedAvgTDPct'], df['BlueAvgTDPct']]).mean()
}

striker_metrics = {
    'AvgSigStrLanded': pd.concat([df['RedAvgSigStrLanded'], df['BlueAvgSigStrLanded']]).mean(),
    'AvgSigStrPct': pd.concat([df['RedAvgSigStrPct'], df['BlueAvgSigStrPct']]).mean(),
    'WinsByKO': pd.concat([df['RedWinsByKO'], df['BlueWinsByKO']]).mean()
}

for metric, values in grappler_metrics.items():
    print(f"Average for {metric}: {values}")
for metric, values in striker_metrics.items():
    print(f"Average for {metric}: {values}")
for metric, values in wrestler_metrics.items():
    print(f"Average for {metric}: {values}")

df['RedIsGrappler'] = ((df['RedAvgSubAtt'] > grappler_metrics['AvgSubAtt']) & (df['RedWinsBySubmission'] > grappler_metrics['WinsBySubmission'])).astype(int)
df['BlueIsGrappler'] = ((df['BlueAvgSubAtt'] > grappler_metrics['AvgSubAtt']) & (df['BlueWinsBySubmission'] > grappler_metrics['WinsBySubmission'])).astype(int)
df['RedIsWrestler'] = ((df['RedAvgTDLanded'] > wrestler_metrics['AvgTDLanded']) & (df['RedAvgTDPct'] > wrestler_metrics['AvgTDPct'])).astype(int)
df['BlueIsWrestler'] = ((df['BlueAvgTDLanded'] > wrestler_metrics['AvgTDLanded']) & (df['BlueAvgTDPct'] > wrestler_metrics['AvgTDPct'])).astype(int)
df['RedIsStriker'] = ((df['RedAvgSigStrLanded'] > striker_metrics['AvgSigStrLanded']) & (df['RedAvgSigStrPct'] > striker_metrics['AvgSigStrPct']) & (df['RedWinsByKO'] > striker_metrics['WinsByKO'])).astype(int)
df['BlueIsStriker'] = ((df['BlueAvgSigStrLanded'] > striker_metrics['AvgSigStrLanded']) & (df['BlueAvgSigStrPct'] > striker_metrics['AvgSigStrPct']) & (df['BlueWinsByKO'] > striker_metrics['WinsByKO'])).astype(int)

In [ ]:
df = df.dropna()
no_fights_mask = (df["RedWins"] == 0) & (df["RedLosses"] == 0) & (df["RedDraws"] == 0) & \
                 (df["BlueWins"] == 0) & (df["BlueLosses"] == 0) & (df["BlueDraws"] == 0)

df = df.drop(df[no_fights_mask].index)


# index_to_drop = df[df['ReachDif'] >= 70].index
# df = df.drop(index_to_drop)

# index_to_drop = df[df['ReachDif'] <= -70].index
# df = df.drop(index_to_drop)


print(f"{len(df)} rows in the df. That means that there should be {(len(df) * 2)} rows in the corr_df")
corr_df =  df.copy()

differentials = [c for c in df.columns if "Dif" in c]
red_cols = [c for c in df.columns if not c.startswith("Blue") and not c.startswith("B")]
blue_cols = [c for c in df.columns if not c.startswith("Red") and (not c.startswith("R") or c == "ReachDif")]

red_df = df[red_cols].copy()
red_df["Win"] = (df["Winner"] == 1).astype(int)
red_df["Loss"] = (df["Winner"] == 0).astype(int)
red_df["isred"] = 1
red_df["isblue"] = 0


print(differentials)

blue_df = df[blue_cols].copy()
blue_df["Win"] = (df["Winner"] == 0).astype(int)
blue_df["Loss"] = (df["Winner"] == 1).astype(int)
blue_df["isred"] = 0
blue_df["isblue"] = 1


def clean_red_name(c):
    if c.startswith("Red"):
        return c.replace("Red", "", 1)
    elif c.startswith("R") and c != "ReachDif":
        return c[1:]
    return c

def clean_blue_name(c):
    if c.startswith("Blue"):
        return c.replace("Blue", "", 1)
    elif c.startswith("B"):
        return c[1:]
    return c

red_df.columns = [clean_red_name(c) for c in red_df.columns]
blue_df.columns = [clean_blue_name(c) for c in blue_df.columns]


for c in differentials:
    red_df[c] = df[c] * -1


corr_df = pd.concat([red_df, blue_df], ignore_index=True)
print(f"{len(corr_df)} rows in the corr_df")

num_rows_with_nan = corr_df.isna().any(axis=1).sum()
print(num_rows_with_nan)
corr_df.head()

In [ ]:
rows_with_nan = corr_df[corr_df.isna().any(axis=1)]
print(rows_with_nan)

In [ ]:
counts = corr_df['Win'].value_counts()
print(counts)

In [ ]:

corr_df = corr_df.drop(columns=["Fighter", "WeightClass", "Country", "Location", "ExpectedValue", "UDec", "SDec", "Sub", "KO/TKO", "Winner"])
num_rows_with_nan = corr_df.isna().any(axis=1).sum()
print(num_rows_with_nan)

In [ ]:
def binAge(age):
  if age < 25:
    return 1
  elif age < 32:
    return 2
  elif age < 37:
    return 3
  else:
    return 4


corr_df["Age_Bin"] = corr_df.apply(lambda row : binAge(row["Age"]), axis=1)
corr_df.head()

num_rows_with_nan = corr_df.isna().any(axis=1).sum()
print(num_rows_with_nan)

In [ ]:
def binOdds(odds):
  if odds < -285:
    return 2
  elif odds < 0:
    return 1
  elif odds > 200:
    return -1
  else:
    return -2


corr_df["Odds_Binned"] = corr_df.apply(lambda row : binOdds(row["Odds"]), axis=1)
corr_df.head()

num_rows_with_nan = corr_df.isna().any(axis=1).sum()
print(num_rows_with_nan)

# Inference
Now that the Data from project one has been transferred over, its time to evaluate the work that was done in the last project.

In the last project I made multiple synthetic features. Now I am going to use logistic regression as a model on that data to see which synthetic features were the best and which synthetic features didn't add much to the model.

The Features that I will be testing are:
- WeightClass Rank
- SouthPaw
- Orthodox
- differentStance
- weight_class_num
- IsGrappler
- IsWrestler
- IsStriker
- isRed
- isBlue
- LastFightWasKOed


## Baseline
Before we evaluate any of the features, we need a control to see how good the logistic regression will do before any of my synthetic features are added.

In [ ]:
corr_df['Win'].value_counts()


In [ ]:
# Remove the loss collumn because that would be a pretty influential weight
corr_df.drop(columns=["Loss"], inplace=True)


In [ ]:
invalid_win_rows = corr_df[~corr_df['Win'].isin([0, 1])]

# Print the rows
print(invalid_win_rows)

In [ ]:
columns_to_drop = [
    "FinishRound",
    "FinishRoundTime",
    "TotalFightTimeSecs"
]

corr_df = corr_df.drop(columns=columns_to_drop)

In [ ]:
corr_df.drop(columns=["Date"], inplace=True)


In order to make a baseline I need to make a copy of my df, corr_df and from that copy drop all of the synthetic features.

In [ ]:
control_df = corr_df.copy()
control_df['Win'].value_counts()

columns_to_drop = [
    "WeightClassRank",
    "Southpaw",
    "Othodox",
    "differentStance",
    "weight_class_num",
    "IsGrappler",
    "IsWrestler",
    "IsStriker",
    "isred",
    "isblue",
    "LastFightWasKOed"
]

control_df = control_df.drop(columns=columns_to_drop)

control_df.head()

# Total number of NaNs in the whole DataFrame
total_nans = control_df.isna().sum().sum()
print(f"Total NaNs in the control_df: {total_nans}")


total_nans = corr_df.isna().sum().sum()
print(f"Total NaNs in the corr_Df: {total_nans}")

I then need to add the code for the logistic regression.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Drop rows with NaN values from control_df before splitting
#control_df_cleaned = control_df.dropna()
control_df_cleaned = control_df
X = control_df_cleaned.drop(columns=["Win"])
y = control_df_cleaned["Win"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=50000) # Increased max_iter for convergence

model.fit(X_train, y_train)

print("Logistic Regression model trained successfully!")

Then I need to train the model and look at its accuracy, precision and recall.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

# Predict on the training set
y_train_pred = model.predict(X_train)
control_train_accuracy = accuracy_score(y_train, y_train_pred)
control_train_precision = precision_score(y_train, y_train_pred)
control_train_recall = recall_score(y_train, y_train_pred)
print(f"Training Accuracy: {control_train_accuracy:.4f}")
print(f"Training Precision: {control_train_precision:.4f}")
print(f"Training Recall: {control_train_recall:.4f}")

# Predict on the testing set
y_test_pred = model.predict(X_test)
control_test_accuracy = accuracy_score(y_test, y_test_pred)
control_test_precision = precision_score(y_test, y_test_pred)
control_test_recall = recall_score(y_test, y_test_pred)

print("\n\n\n")
print(f"Testing Accuracy: {control_test_accuracy:.4f}")
print(f"Testing Precision: {control_test_precision:.4f}")
print(f"Testing Recall: {control_test_recall:.4f}")

Lets analyze the results of the baseline.


This is an interesting side of the report. There was a major bug that resulted in there being much fewer losers than there were winners in the dataframe. This lead to my model predicting positve. I will later in the report mention why the bug was occuring and what happend. But right here is my explanation of what I was thinking at the time.


We were getting a recall of 79% of the positives which was good, but our precision was 63%. This indicated that we had a lot of false potives and that our model is overpredicting postive. This would've been a large problem because I want every prediction to be correct. This is not a cancer diagnosis where it is more important that I catch all positives than for a situation where I want to pick the winner of a fight, so predicting winner is just as important as predicnting the loser.

Since I had not yet checked if my data was ballanced, I thought that this had to be an issue of threshold. I figured that my threshold had to be too low and that I would need to increase it.

I then made a function find_best_threshold which would start at a given threshold and would increase the threshold by a step amount storing the accuracy, precision, and recall at each step. From there I tried to maximize the precision because I thought that having my correct predictions be correct mattered the most. Later on I realize that this is flawed and I correct myself, but that is what I did at the given time.  

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score


def find_best_threshold(starting_threshold, model_output, y_true, step=0.01):
  results = []

  for threshold in np.arange(starting_threshold, 1.0, step):
    y_pred_custom = (model_output > threshold).astype(int)

    accuracy = accuracy_score(y_true, y_pred_custom)
    precision = precision_score(y_true, y_pred_custom, zero_division=0)
    recall = recall_score(y_true, y_pred_custom, zero_division=0)

    apDiff = abs(precision - recall)

    results.append([threshold, accuracy, precision, recall, apDiff])

  results_df = pd.DataFrame(results, columns=['Threshold', 'Accuracy', 'Precision', 'Recall', 'APDiff'])
  results_df = results_df[results_df['Precision'] >= .5]
  results_df = results_df[results_df['Accuracy'] >= 0.5]
  results_df = results_df[results_df['Recall'] >= 0.5]

  results_df = results_df.sort_values(by='Precision', ascending=False)



  return results_df.head()

In [ ]:
find_best_threshold(0, model.predict_proba(X_train)[:, 1], y_train, .001)

Because my data was so skewed then, I was able to get precision and recall very close together and it did not hurt my accuracy much on the tested data. As you can see now, my accuracy did not change, but there was still a large discrepancy with the Recall. I should've known that all this did was make my model predict less positives and only pick it if it was pretty sure it was positive. I put a minimum requirement of .5 Accuracy and .5 Recall because I didn't want the model to punt on Recal and Accuracy compeltely to maximize Precision...but as you can see the model just punted on it as much as I let it.

With the old data, I found a best threshold of .575.

In [ ]:
test_results = model.predict_proba(X_test)[:, 1]
y_pred_custom = (test_results > 0.575).astype(int)

def testModel(model, X_test, y_test, threshold):
  test_results = model.predict_proba(X_test)[:, 1]
  y_pred_custom = (test_results > threshold).astype(int)
  accuracy = accuracy_score(y_test, y_pred_custom)
  precision = precision_score(y_test, y_pred_custom)
  recall = recall_score(y_test, y_pred_custom)

  print(f"Accuracy: {accuracy:.4f}")
  print(f"Precision: {precision:.4f}")
  print(f"Recall: {recall:.4f}")

testModel(model, X_test, y_test, .5)


It gave me results that were all very close together and I was pleased enough with the results that I took it as a baseline and continued to evaluating the synthetic features.

##LastFightKOed
Now that the baseline was set, I stared looking at the synthetic features. Starting first with LastFightKOed. For this I did the same thing as I had before, but instead started wutg the Threshold of .575

In [ ]:
def myLR(df, threshold, numIterations=1000):

  df_cleaned = df

  X = df_cleaned.drop(columns=["Win"])
  y = df_cleaned["Win"]

  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

  model = make_pipeline(
      StandardScaler(),               # scales numeric features to mean=0, std=1
      LogisticRegression(solver='lbfgs', max_iter=numIterations)
  )

  model.fit(X_train, y_train)


  train_results = model.predict_proba(X_train)[:, 1]
  y_pred_custom_train = (train_results > threshold).astype(int)

  print(f"Training Accuracy: {accuracy_score(y_train, y_pred_custom_train):.4f}")
  print(f"Training Precision: {precision_score(y_train, y_pred_custom_train):.4f}")
  print(f"Training Recall: {recall_score(y_train, y_pred_custom_train):.4f}")

  test_results = model.predict_proba(X_test)[:, 1]
  y_pred_custom_test = (test_results > threshold).astype(int)

  print("\n\n\n")
  print(f"Testing Accuracy: {accuracy_score(y_test, y_pred_custom_test):.4f}")
  print(f"Testing Precision: {precision_score(y_test, y_pred_custom_test):.4f}")
  print(f"Testing Recall: {recall_score(y_test, y_pred_custom_test):.4f}")

  return [model, X_test, y_test]

In [ ]:
columns_to_drop = [
    "WeightClassRank",
    "Southpaw",
    "Othodox",
    "differentStance",
    "weight_class_num",
    "IsGrappler",
    "IsWrestler",
    "IsStriker",
    "isred",
    "isblue",
]

lastFightKoed_df = corr_df.copy(deep=True)

lastFightKoed_df = lastFightKoed_df.drop(columns=columns_to_drop)


output = myLR(lastFightKoed_df, .575, 500000)

I was pleased with these results as it caused a really good percision and a similar accuracy. However I did the same thing where I tried to maximize the precision.

In [ ]:
find_best_threshold(0, output[0].predict_proba(output[1])[:, 1], output[2], .001)

In [ ]:
testModel(output[0], output[1], output[2], 0.568)

On the old data, and the current data this showed me that lastFightKOed was a significant feature as it increased the precision bt 5%.

##Changing it up
Then I met with you and we discussed Neural Networks. At that point I stopped looking at Logistic Regression and fully transissioned to working on Neural Netowrks.

I started again by finding a baseline. However I wanted to find some other baselines first to compare my model overall with. I was pretty happy with my model as it was doing really well. The first baseline I found was what would it be if I always picked the Red fighter. Project 1 showed me that that fighter won very often.

In [ ]:

counts = corr_df['Win'].value_counts()
print(counts)

In [ ]:
def evaluate_isRed_rule(df: pd.DataFrame):
    feature = "isred"
    target = "Win"

    # Safety checks
    if feature not in df.columns:
        raise ValueError("Column 'isRed' not found in DataFrame.")
    if target not in df.columns:
        raise ValueError("Column 'Win' not found in DataFrame.")

    # Convert to boolean just in case

    feature_values = df[feature].astype(bool)
    actual_values = df[target]

    # Always predict Win = 1 when isRed == True
    y_pred = feature_values.astype(int)

    # ---- Confusion Matrix ----
    TP = ((y_pred == 1) & (actual_values == 1)).sum()
    FP = ((y_pred == 1) & (actual_values == 0)).sum()
    TN = ((y_pred == 0) & (actual_values == 0)).sum()
    FN = ((y_pred == 0) & (actual_values == 1)).sum()

    # ---- Metrics ----
    accuracy = (TP + TN) / len(df)
    precision = TP / (TP + FP) if (TP + FP) != 0 else 0
    recall = TP / (TP + FN) if (TP + FN) != 0 else 0

    # ---- Print Results ----
    print("Feature Rule: Always predict Win when isRed == True")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print("\nConfusion Matrix Values:")
    print(f"TP: {TP}, FP: {FP}, TN: {TN}, FN: {FN}")

evaluate_isRed_rule(corr_df)

This gave me the goal of beating 57%, if I could produce a model that beat 57% that meant that I could beat picking the Red fighter every single time. .


This gave me another idea. I have the oods, what would my success be if I always picked the fighter who was predicted to win.

In [ ]:
def evaluate_negative_odds_rule(df: pd.DataFrame):
    odds_col = "Odds"
    target = "Win"

    # ---- Safety Checks ----
    if odds_col not in df.columns:
        raise ValueError("Column 'Odds' not found in DataFrame.")
    if target not in df.columns:
        raise ValueError("Column 'Win' not found in DataFrame.")

    actual_values = df[target]

    # ----------------------------------------
    # PREDICTION RULE:
    # If Odds < 0  → Predict Win (1)
    # If Odds >= 0 → Predict Loss (0)
    # ----------------------------------------

    y_pred = (df[odds_col] < 0).astype(int)

    # ----------------------------------------
    # CONFUSION MATRIX COMPONENTS
    # ----------------------------------------

    TP = ((y_pred == 1) & (actual_values == 1)).sum()
    FP = ((y_pred == 1) & (actual_values == 0)).sum()
    TN = ((y_pred == 0) & (actual_values == 0)).sum()
    FN = ((y_pred == 0) & (actual_values == 1)).sum()

    # ----------------------------------------
    # METRICS
    # ----------------------------------------

    accuracy = (TP + TN) / len(df)
    precision = TP / (TP + FP) if (TP + FP) != 0 else 0
    recall = TP / (TP + FN) if (TP + FN) != 0 else 0

    # ----------------------------------------
    # PRINT RESULTS
    # ----------------------------------------

    print("Rule: Always predict Win when Odds < 0\n")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")

    print("\nConfusion Matrix:")
    print(f"TP: {TP}")
    print(f"FP: {FP}")
    print(f"TN: {TN}")
    print(f"FN: {FN}")

evaluate_negative_odds_rule(corr_df)

This was very unfortunate. I thought that my model was doing really well, but what was probably happening was that it was putting a large weight on the odds becuase it performed about as good as just picking the odds every time.

This was disapointing, but also should've been expected because the odds makers are professionals with a lot more experience than I do.

Because of this, I decided that using the odds would be cheating and would hide how significant my features actually were. Because of that, I decided to not remove odds from the list of synthetic features. The goal became how close to the sportsbooks could I get my model.

The frist thing we need to do is remove Odds from our control_df

In [ ]:
control_df = control_df.drop(columns=["Odds","Odds_Binned"])
print(f"Original size: {len(control_df)}")
# control_df = control_df.dropna()
# print(f"After dropping NaN: {len(control_df)}")


In [ ]:
control_df.head()

Now I began the task of building the Neural Network. To do this I followed the example that we worked on in class.

I will walk you through what I did.

First import all of the necessary imports.

In [ ]:
## Standard libraries
import os
import math
import numpy as np
import time

## Imports for plotting
import matplotlib.pyplot as plt
%matplotlib inline
from IPython.display import set_matplotlib_formats
set_matplotlib_formats('svg', 'pdf') # For export
from matplotlib.colors import to_rgba
import seaborn as sns
sns.set()

## Progress bar
from tqdm.notebook import tqdm

import torch

# Use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data

Then we need to make the class for the neural network which takes the number of inputs, the number of hidden nodes, and the number of outputs. And other methods to help with training.

In [ ]:
class UFCNN(nn.Module):
    def __init__(self, num_inputs, num_hidden, num_outputs):
        super().__init__()

        self.linear1 = nn.Linear(num_inputs, num_hidden)
        self.act_fn = nn.ReLU()
        self.linear2 = nn.Linear(num_hidden, num_outputs)

    def forward(self, x):
        # Perform the calculation of the model to determine the prediction
        x = self.linear1(x)
        x = self.act_fn(x)
        x = self.linear2(x)
        return x

Then we need to make the class that will create and control the dataset for the Neural Network in order to allow the nerual network to train the dataset.

In [ ]:
class UFCDataset(data.Dataset):
  def __init__(self, df: "pd.DataFrame", feature_cols: list, label_col: str):
        super().__init__()
        self.features = torch.tensor(df[feature_cols].values, dtype=torch.float32)
        self.labels = torch.tensor(df[label_col].values, dtype=torch.float32)
        self.size = len(df)

  def __len__(self):
      return self.size

  def __getitem__(self, idx):
      return self.features[idx], self.labels[idx]

Now we must split our dataset into training data and teting data.

In [ ]:
#Split dataset into train and test
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(control_df, test_size=0.2, random_state=42)

Now we get all of the feature cols to feed into the model. We do this by getting all of the cols in control_df and removing the Win col.

In [ ]:
feature_cols = control_df.columns.tolist()
feature_cols.remove("Win")




This was code thatI wrote to try to find where the model was going wrong when I was solving the issue of an unbalanced df.

In [ ]:
# See NaN count per column
nan_counts = control_df[feature_cols].isna().sum()
print("NaN counts per column:")
print(nan_counts[nan_counts > 0].sort_values(ascending=False))

# Get the percentage of NaN in each column
nan_percentages = (control_df[feature_cols].isna().sum() / len(control_df)) * 100
print("\nNaN percentages per column:")
print(nan_percentages[nan_percentages > 0].sort_values(ascending=False))

# Find which column has the most NaNs
most_common_col = nan_counts.idxmax()
print(f"\nColumn with most NaNs: {most_common_col} ({nan_counts[most_common_col]} NaNs)")

# See some example rows with NaN values
print("\nSample rows with NaN values:")
print(control_df[control_df[feature_cols].isna().any(axis=1)].head(10))

# Count how many rows have at least one NaN
rows_with_nan = control_df[feature_cols].isna().any(axis=1).sum()
print(f"\nTotal rows with at least one NaN: {rows_with_nan}")
print(f"Total rows: {len(control_df)}")
print(f"Percentage of rows affected: {(rows_with_nan/len(control_df))*100:.2f}%")

When I was running the model earlier it was not training well because the data was too different and per gemini's suggestions I used StandardScaler to scale the data.

In [ ]:

scaler = StandardScaler()
train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
test_df[feature_cols] = scaler.transform(test_df[feature_cols])


In [ ]:
print(train_df[feature_cols].describe())
print(test_df[feature_cols].describe())


Now we need to seperate the train and test data into separate dataset instances

In [ ]:

train_dataset = UFCDataset(train_df, feature_cols=feature_cols, label_col="Win")
train_data_loader = data.DataLoader(train_dataset, batch_size=16, shuffle=True)

test_dataset = UFCDataset(test_df, feature_cols=feature_cols, label_col="Win")
test_data_loader = data.DataLoader(test_dataset, batch_size=16, shuffle=False)


Now I we are going to make an instance of the UFCNN class as our model and give it num_inputs equal to the number of features we have, and num_hiddes equal to twice the number of features we have. We only want 1 output.

In [ ]:

num_inputs = len(feature_cols)
num_hidden = 2 * num_inputs

model = UFCNN(num_inputs=num_inputs, num_hidden=num_hidden, num_outputs=1)


Then we need to define our loss module which tells the model how wrong my predictions are and the optimizer which updates the weights after each iteration.

In [ ]:
loss_module = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

Now we need to make the function that trains the data, this takes

In [ ]:
def train_model(model, optimizer, data_loader, loss_module, num_epochs=100, plotter=None):
    # Set model to train mode
    model.train()

    # Training loop
    for epoch in tqdm(range(num_epochs)):
        for data_inputs, data_labels in data_loader:

            ## Step 2: Run the model on the input data
            preds = model(data_inputs)
            preds = preds.squeeze(dim=1) # Output is [Batch size, 1], but we want [Batch size]

            ## Step 3: Calculate the loss
            loss = loss_module(preds, data_labels.float())

            ## Step 4: Perform backpropagation
            # Before calculating the gradients, we need to ensure that they are all zero.
            # The gradients would not be overwritten, but actually added to the existing ones.
            optimizer.zero_grad()
            # Perform backpropagation
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            ## Step 5: Update the parameters
            optimizer.step()

Now we must train the model and save the resulting weights so that we can use them later.

In [ ]:
train_model(model, optimizer, train_data_loader, loss_module)
state_dict = model.state_dict()
print(state_dict)
torch.save(state_dict, "baseline.tar")

Now we are going to test that the model was saved correctly and that we can load it into a new model and that it will have the same weights. This does not train the model again, it just loads it. This will be useful later for comparing all of the models.

In [ ]:
# Load state dict from the disk (make sure it is the same name as above)
state_dict = torch.load("baseline.tar")

# Create a new model and load the state
new_model = UFCNN(num_inputs=num_inputs, num_hidden=num_hidden, num_outputs=1)
new_model.load_state_dict(state_dict)

# Verify that the parameters are the same
print("Original model\n", model.state_dict())
print("\nLoaded model\n", new_model.state_dict())

Now we need to evaluate our model on the test data. To do that we need to convert the predictions to probabilities and classify everything and then compare it to the actual results. Then we will calculate our accuracy, precision, and recall.

In [ ]:
def eval_model(model, data_loader):
    model.eval()  # Set model to evaluation mode

    TP = 0.0
    FP = 0.0
    FN = 0.0
    TN = 0.0

    with torch.no_grad():  # Turn off gradients
        for data_inputs, data_labels in data_loader:


            # Forward pass
            preds = model(data_inputs)
            preds = preds.squeeze(dim=1)
            preds = torch.sigmoid(preds)  # Convert logits → probabilities

            # Convert probabilities to 0/1 predictions
            pred_labels = (preds >= 0.5).long()

            # ---- CONFUSION MATRIX COUNTS ----
            TP += ((pred_labels == 1) & (data_labels == 1)).sum().item()
            FP += ((pred_labels == 1) & (data_labels == 0)).sum().item()
            FN += ((pred_labels == 0) & (data_labels == 1)).sum().item()
            TN += ((pred_labels == 0) & (data_labels == 0)).sum().item()

    # ---- METRICS ----
    accuracy = (TP + TN) / (TP + TN + FP + FN)

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0

    print(f"Accuracy : {100.0 * accuracy:.2f}%")
    print(f"Precision: {100.0 * precision:.2f}%")
    print(f"Recall   : {100.0 * recall:.2f}%")

    return accuracy, precision, recall




In [ ]:
print("Training Results")
baseline_train_accuracy, baseline_train_precision, baseline_train_recall = eval_model(model, train_data_loader)
print("\n\n")
print("Test Results")
baseline_test_accuracy, baseline_test_precision, baseline_test_recall = eval_model(model, test_data_loader)

This is where I noticed that there was an issue. I had a a Recall of 83% which really raised the red flags. This caused the manhunt that I will explain in the follwing paragraph.

In [ ]:
counts = control_df['Win'].value_counts()
print(counts)


This looks good, but it didn't used to. I spent an hour and a half trying to figure out why, but I had 57% winners and 43% losers. This was not ok because my modle could cheat and predict winners more often than losers. I was keyed into this by the 83% recall. After scowering the code...it turns out to be becasue of two problems.
Firstly I removed the the NaN rows from the corr_df not the df, so it kept the fighters from fights who had NaN values.

Secondly, and this was the much bigger problem. When I removed fighters who had not fought yet in the ufc, it removed a lot more losers than winners. And I did this to the corr_df instead of the main df. I had to go back and fix this, but now I have much better values. Here is the line of code that broke everything.

no_fights_mask = (corr_df["Wins"] == 0) & (corr_df["Losses"] == 0) & (corr_df["Draws"] == 0)

corr_df = corr_df.drop(corr_df[no_fights_mask].index)

Update: That wasn't the only line that was breaking it. I did the same thing when I tried to fix the ridiculous reach advantage. I now have gone back and made it so that if any row in the corr_df is to be dropped, it will be done to the df before corr_df is made.

Update^2: That turned out to not be the issue. It turns out that I had a major bug in my code with this line when making corr_df.

blue_cols = [c for c in df.columns if not c.startswith("Red") and (not c.startswith("R"))]

This never added the ReachDif column to any blue fighters making all blue fighters null. This was a huge mistake, but I fixed it relatively easily. Now there are no more NaN values and an equal amount of wins and losses. The previous issues were problematic, but the large problem came from that line of code right there.

I solved it with. blue_cols = [c for c in df.columns if not c.startswith("Red") and (not c.startswith("R") or c == "ReachDif")]



Since I care equally about positive and negative classifications, and since there is now a 50/50 split of positives and negatives. My goal is to optimize the accuracy since accuracy is what is most important for me.

##Last Fight KO
There turns out to be a major issue with how my NN was set up that I will explain later, but for right now I am going to go back to the report style that I was doing before all of the revisionist history. Not all of my features helped, but I now know why. Unfortunately I do not have the time to run them all again simply because the time it takes for me to run my NN is way too long at this point in the project and I need to move on to other things.

Now that we have our baseline. It should be relatively easy to run these again. with the new feature. As with the the Logistic Regression, the first synthetic feature we will try is LastFightKoed. All I need to do to run this is copy most of the same code, but use a different dataset than control_df.

In [ ]:
columns_to_drop = [
    "WeightClassRank",
    "Southpaw",
    "Othodox",
    "differentStance",
    "weight_class_num",
    "IsGrappler",
    "IsWrestler",
    "IsStriker",
    "isred",
    "isblue",
    "Odds",
    "Odds_Binned"
]
lastFightKoed_df = corr_df.copy(deep=True)

lastFightKoed_df = lastFightKoed_df.drop(columns=columns_to_drop)


In [ ]:

def runNN(df):
  train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)


  feature_cols = df.columns.tolist()
  feature_cols.remove("Win")

  scaler = StandardScaler()
  train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
  test_df[feature_cols] = scaler.transform(test_df[feature_cols])



  train_dataset = UFCDataset(train_df, feature_cols=feature_cols, label_col="Win")
  train_data_loader = data.DataLoader(train_dataset, batch_size=16, shuffle=True)

  test_dataset = UFCDataset(test_df, feature_cols=feature_cols, label_col="Win")
  test_data_loader = data.DataLoader(test_dataset, batch_size=16, shuffle=False)


  num_inputs = len(feature_cols)
  num_hidden = 2 * num_inputs

  model = UFCNN(num_inputs=num_inputs, num_hidden=num_hidden, num_outputs=1)


  loss_module = nn.BCEWithLogitsLoss()
  optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

  train_model(model, optimizer, train_data_loader, loss_module)
  state_dict = model.state_dict()
  print(state_dict)
  torch.save(state_dict, "LFKO.tar")

  print("Training Results")
  baseline_train_accuracy, baseline_train_precision, baseline_train_recall = eval_model(model, train_data_loader)
  print("\n\n")
  print("Test Results")
  baseline_test_accuracy, baseline_test_precision, baseline_test_recall = eval_model(model, test_data_loader)

  return model, baseline_train_accuracy, baseline_train_precision, baseline_train_recall, train_data_loader, test_data_loader

In [ ]:
runNN(lastFightKoed_df)

LastFightWasKoed is a useful synthetic feature. It increases the accuracy of the model by .3% which is relatively useful. My guess as of right now is that it always marks Loss if the fighter was knocked out because we see a 3% in Recall from the baseline Recall. This means that the number of negatives that we are predicting is increasing.



##SouthPaw


The next feature we will look at is SouthPaw, I don't expect that this one alone will increase the accuracy of the model much, however I do expect that when I combine it with differnce stance it will. I am going to run this model twice, once where the only synthetic feature is Southpaw, and a second time where I add Southpaw and LastFightKoed. First I will do just Southpaw.


In [ ]:
columns_to_drop = [
    "WeightClassRank",
    "Othodox",
    "differentStance",
    "weight_class_num",
    "IsGrappler",
    "IsWrestler",
    "IsStriker",
    "isred",
    "isblue",
    "Odds",
    "Odds_Binned",
    "LastFightWasKOed",
]

south_paw_df = corr_df.copy(deep=True)
south_paw_df = south_paw_df.drop(columns=columns_to_drop)

runNN(south_paw_df)




I'm actually very surprised by this, adding the southPaw feature decreases the accuracy by .2. My guess as to why this is happening, is that Southpaws are getting marked as Win more often. My reasoning for this is that the recall increases meaning that the amount of correct predictions is increasing. So if accuracy decreasses and Recall increases, the number of Win predictions must increase. Notice also that the training perfomed much better than the test insinuating possible overfitting.


Now lets combine both the southpaw feature and the lastfight koed feature.

In [ ]:
columns_to_drop = [
    "WeightClassRank",
    "Othodox",
    "differentStance",
    "weight_class_num",
    "IsGrappler",
    "IsWrestler",
    "IsStriker",
    "isred",
    "isblue",
    "Odds",
    "Odds_Binned",
]


cummulative_df = corr_df.copy(deep=True)
cummulative_df = cummulative_df.drop(columns=columns_to_drop)

runNN(cummulative_df)

Adding both features together increased the accuracy overall, not by much, but it is still an improvement over the baseline.

##DifferentStance

I expect that this one alone will not change that much from the baseline. Letting them know that there is a fight where there is a different stance alone doesn't seem relevant to me, however when I combine it with the other synthetic features namely, is SouthPaw. I expect that it will increase the accuracy significantly.


First we will look at just the differentStance.

In [ ]:
columns_to_drop = [
    "WeightClassRank",
    "Othodox",
    "weight_class_num",
    "IsGrappler",
    "IsWrestler",
    "IsStriker",
    "isred",
    "isblue",
    "Odds",
    "Odds_Binned",
    "LastFightWasKOed",
    "Southpaw"
]

diff_stance_df = corr_df.copy(deep=True)
diff_stance_df = diff_stance_df.drop(columns=columns_to_drop)

runNN(diff_stance_df)


I was very suprised that this decreased the accuracy of the training data and the testing data.

This should not be the case, but as you will see later, I foudn out why this was occuring. This happens for a lot of them. For this part of the report you get to wittness my distress and confusion as to what was going on.

In [ ]:
columns_to_drop = [
    "WeightClassRank",
    "Othodox",
    "weight_class_num",
    "IsGrappler",
    "IsWrestler",
    "IsStriker",
    "isred",
    "isblue",
    "Odds",
    "Odds_Binned",
]


cummulative_df = corr_df.copy(deep=True)
cummulative_df = cummulative_df.drop(columns=columns_to_drop)

runNN(cummulative_df)

This is very disapointing. I was expecting that it would increase the accuracy by a decent amount, but it actyally decreased the accuracy of the model from the previous cummulative feature df. I will continue with this process and at the end decide if I want to keep it or not.

## WeightClass Rank
I had added the Orthodox feature, but now that I am thinking about it, Orthodox and isBlue are both features that are already represented in Southpaw and isred, so neither of them add anything to the model at all. For this reason I will not look at isBlue or Orthodox.

The next feature that we will be looking at is, WeightClass Rank. I expect that this one will increase the accuracy of the model by a decent bit. The Model already has betterRank, however I think knowing the actual rank should mean something.

In [ ]:
columns_to_drop = [
    "Othodox",
    "weight_class_num",
    "IsGrappler",
    "IsWrestler",
    "IsStriker",
    "differentStance",
    "isred",
    "isblue",
    "Odds",
    "Odds_Binned",
    "LastFightWasKOed",
    "Southpaw"
]

single_feature_df = corr_df.copy(deep=True)
single_feature_df = single_feature_df.drop(columns=columns_to_drop)

runNN(single_feature_df)

This one cause heavy overfitting, it caused the accuracy fo the training data to increase by a good amount however it decreased the accuracy again. This shows that it is probably not a good synthetic feature.

The next one that we will look at is the cumulative model's results.

In [ ]:
columns_to_drop = [
    "Othodox",
    "weight_class_num",
    "IsGrappler",
    "IsWrestler",
    "IsStriker",
    "isred",
    "isblue",
    "Odds",
    "Odds_Binned",
]


cummulative_df = corr_df.copy(deep=True)
cummulative_df = cummulative_df.drop(columns=columns_to_drop)

runNN(cummulative_df)

This was NN doing their magic, even though the weightclassRank caused overfitting alone, it ended up causing a .3% increase in accuracy when it was combined with the other features.

##Weight_Class_Number

I have high expectations for this result, which means that if curren't trends continue, it will do nothing but make the model overfit. I think that knowing what the weightclass is will help draw a lot of connections and should increase the accuracy.

In [ ]:
columns_to_drop = [
    "Othodox",
    "WeightClassRank",
    "IsGrappler",
    "IsWrestler",
    "IsStriker",
    "differentStance",
    "isred",
    "isblue",
    "Odds",
    "Odds_Binned",
    "LastFightWasKOed",
    "Southpaw"
]

single_feature_df = corr_df.copy(deep=True)
single_feature_df = single_feature_df.drop(columns=columns_to_drop)

runNN(single_feature_df)

Well...This did almost nothing. At first I thought this was shocking...then I realized that we are giving them the fighters weights, so the model is already calculating based on this information. I'll see if it makes the cummulative model better, but I expect this to be useless.

In [ ]:
columns_to_drop = [

    "IsGrappler",
    "IsWrestler",
    "IsStriker",
    "isred",



    "isblue",
    "Odds",
    "Othodox",
    "Odds_Binned",
]


cummulative_df = corr_df.copy(deep=True)
cummulative_df = cummulative_df.drop(columns=columns_to_drop)

runNN(cummulative_df)

This barelly increased the model's accuracy in testing. It increased it by .12%. I am very confident in my theory that weightclass does matter, but since the model already has the fighters weight it was already accounting for it.

##IsGrappler
This is another feeature I had decently high expectation. It had decently high correlation for winning in phase 1 so I expect it to be relevant here.

In [ ]:
columns_to_drop = [
    "Othodox",
    "WeightClassRank",
    "weight_class_num",
    "IsWrestler",
    "IsStriker",
    "differentStance",
    "isred",
    "isblue",
    "Odds",
    "Odds_Binned",
    "LastFightWasKOed",
    "Southpaw"
]

single_feature_df = corr_df.copy(deep=True)
single_feature_df = single_feature_df.drop(columns=columns_to_drop)

runNN(single_feature_df)

Alone it did not increase from the baseline much if at all (.03% ) on the training data and only .15 of the testing data.

In [ ]:
columns_to_drop = [

    "IsWrestler",
    "IsStriker",
    "isred",



    "isblue",
    "Odds",
    "Othodox",
    "Odds_Binned",
]


cummulative_df = corr_df.copy(deep=True)
cummulative_df = cummulative_df.drop(columns=columns_to_drop)

runNN(cummulative_df)

While it had little effect alone. isGrappler increased the performance of the whole model by .2% on testing data and .3% on training data. This makes it a useful feature.


##IsWrestler

This is the feature that was the most predictive in my reasearch in project 1...so I have very high expectation for it.

In [ ]:
columns_to_drop = [
    "Othodox",
    "WeightClassRank",
    "weight_class_num",
    "IsGrappler",
    "IsStriker",
    "differentStance",
    "isred",
    "isblue",
    "Odds",
    "Odds_Binned",
    "LastFightWasKOed",
    "Southpaw"
]

single_feature_df = corr_df.copy(deep=True)
single_feature_df = single_feature_df.drop(columns=columns_to_drop)

runNN(single_feature_df)

This implies that alone all it does is cause overfitting. IT icnreases the trainig accuracy by .4% but it causes a decrease in the testing accuracy of .87% making it a very bad feature alone.  

In [ ]:
columns_to_drop = [

    "IsStriker",
    "isred",



    "isblue",
    "Odds",
    "Othodox",
    "Odds_Binned",
]


cummulative_df = corr_df.copy(deep=True)
cummulative_df = cummulative_df.drop(columns=columns_to_drop)

runNN(cummulative_df)

This decreased the accuracy by over 1%. Meaning that all this feature does is cause overfitting.

#IsStriker

This was the least predictive thing in the Project1, I would expect that this would not increase the baseline much.

In [ ]:
columns_to_drop = [
    "Othodox",
    "WeightClassRank",
    "weight_class_num",
    "IsGrappler",
    "IsWrestler",
    "differentStance",
    "isred",
    "isblue",
    "Odds",
    "Odds_Binned",
    "LastFightWasKOed",
    "Southpaw"
]

single_feature_df = corr_df.copy(deep=True)
single_feature_df = single_feature_df.drop(columns=columns_to_drop)

runNN(single_feature_df)

As expected this did not do much. All it did by itself was cause slight overfitting. It slightly increased the training accuracy and decreased the testing accuracy.


Now to see how it effects the overall model.

In [ ]:
columns_to_drop = [
    "isred",



    "isblue",
    "Odds",
    "Othodox",
    "Odds_Binned",
]


cummulative_df = corr_df.copy(deep=True)
cummulative_df = cummulative_df.drop(columns=columns_to_drop)

runNN(cummulative_df)

This was very surprising. It caused huge increase to both training and testing results. It increased the training accuracy by over 1% and it caused a increase in the overall testing accuracy of 1.4%. This feature ended up being very useful in tandem with the other features.

#IsRed
I think this one should greatly increaase the accuracy. isRed was one of the most predictive features in project 1 and if you only pick red fighters you have an accuracy of around 57% I would be shocked if this does not increase the Accuracy.

In [ ]:
columns_to_drop = [
    "Othodox",
    "WeightClassRank",
    "weight_class_num",
    "IsGrappler",
    "IsWrestler",
    "IsStriker",
    "differentStance",
    "isblue",
    "Odds",
    "Odds_Binned",
    "LastFightWasKOed",
    "Southpaw"
]

single_feature_df = corr_df.copy(deep=True)
single_feature_df = single_feature_df.drop(columns=columns_to_drop)

runNN(single_feature_df)

As I expected it increases the accuracy over the baseline by .8% this feature was very predictive.

Its possible it icreases the cumulative results even more.

In [ ]:
columns_to_drop = [
    "isblue",
    "Odds",
    "Othodox",
    "Odds_Binned",
]
cummulative_df = corr_df.copy(deep=True)
cummulative_df = cummulative_df.drop(columns=columns_to_drop)

In [ ]:




model, baseline_train_accuracy, baseline_train_precision, baseline_train_recall, train_data_loader, test_data_loader = runNN(cummulative_df)

This caused a increase in the trainig accuracy of .2 and an increase in the testing accuracy of .5% proving to be very significant.

## Greedy Optimizer
The next thing I wanted to do was go through and figure out what the best combination of features for my nerual network is because there were some features that made the neral network worse. I wrote a Greedy algorithm that finds the best combinations of features and returns early if adding a new feature would make the algorithm worse. What this does is it it tries each feature on the base model tracking the feature that gives the best accuracy. At the end it picks the best one and then continues and repeats. It returns early if adding a new feature makes the algorithm worse.

You told me not to run this, but here is the code I would've ran.

In [ ]:
old_corr_df = corr_df.copy(deep=True)

In [ ]:
from itertools import combinations
from copy import deepcopy


cols_to_drop = [
    "isblue",
    "Odds",
    "Othodox",
    "Odds_Binned"
]


corr_df = corr_df.drop(columns=cols_to_drop)


synthetic_features = [
    "WeightClassRank",
    "weight_class_num",
    "IsGrappler",
    "IsWrestler",
    "IsStriker",
    "isred",
    "differentStance",
    "LastFightWasKOed",
    "Southpaw"
]


def optimize_features_with_cv(df: pd.DataFrame, synthetic_features):
  remaining_features = deepcopy(synthetic_features)
  selected_features = []
  alltime_best_accuracy = 0
  best_actual_model = None

  while len(remaining_features) > 0:
    best_accuracy = 0
    best_feature = []
    best_model = None

    for feature in remaining_features:
      features_to_drop = deepcopy(remaining_features)
      features_to_drop.remove(feature)
      copy_df = df.copy(deep=True)
      copy_df = copy_df.drop(columns=features_to_drop)
      model, accuracy, precision, recall = runNN(copy_df)

      if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_feature = feature
        best_model = model

    if best_accuracy > alltime_best_accuracy:
      alltime_best_accuracy = best_accuracy
      selected_features.append(best_feature)
      remaining_features.remove(best_feature)
      best_actual_model = best_model
      print(best_feature, best_accuracy)
    else:
      return selected_features, alltime_best_accuracy, best_actual_model
  return selected_features, alltime_best_accuracy, best_actual_model





#Finding What went Wrong

Some of the features made my model perform worse instead of better. This should not be the case.
The features that caused a decrease in model performance were:

differentStance
weight_class_rank
Weight_class_num
isWrestler

I am going to run the model again without these features and note how it performs


In [ ]:
columns_to_drop = [
    "differentStance",
    "WeightClassRank",
    "weight_class_num",
    "IsWrestler"
]


extra_df = corr_df.copy(deep=True)
extra_df = extra_df.drop(columns=columns_to_drop)

runNN(extra_df)

So it turns out that even though those features decreased the accuracy of the model in the order that they were added. Overall they proved to be significant.

#Neural Network vs Logistic Regression
Now I am going to compare it with the full Logistic Regression Model to see if my nerual network or my logistic regression model is better

In [ ]:

cummulative_df = corr_df.copy(deep=True)

output = myLR(cummulative_df, .5, 100000)
testModel(output[0], output[1], output[2], 0.5)

So Logistic Regression is outperforming Neural Network...this doesn't make sense and indicates that something is either wrong with my Neural Net or that something can be done to improve it.

I asked Dr. Wolfe what his thoughs were on this because you were unavailable at the time and he suggested trying to use Sigmoid instead of ReLU. He said that this will make it so that one neuron should perform like logistic regression. I am going to try that now. If this does not improve my model then that means there is a bug somewhere because it would not make sense for my Neural Net to underperform logistic regression.

In [ ]:
class UFCNN(nn.Module):
    def __init__(self, num_inputs, num_hidden, num_outputs):
        super().__init__()

        self.linear1 = nn.Linear(num_inputs, num_hidden)
        self.act_fn = nn.Sigmoid()
        self.linear2 = nn.Linear(num_hidden, num_outputs)

    def forward(self, x):
        # Perform the calculation of the model to determine the prediction
        x = self.linear1(x)
        x = self.act_fn(x)
        x = self.linear2(x)
        return x


In [ ]:
model, baseline_train_accuracy, baseline_train_precision, baseline_train_recall, train_data_loader, test_data_loader = runNN(cummulative_df)

Again it underperformed Logistic Regression. At this point I took it to you again thoughroughly confused. And we decided to look at the epochs. I am going to try your live class plot to watch how it converges and see if I was stopping it too early with only 100 epochs.

To do that I need to both add your code and modify my existing NN to print the plots.


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import clear_output
import tensorflow as tf

class LiveLossPlot(tf.keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.losses = []

    def on_epoch_end(self, epoch, train_loss=None):
        self.losses.append(train_loss)

        clear_output(wait=True)   # clears old plot
        plt.plot(self.losses)
        plt.title("Training Loss (updates each epoch)")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.grid(True)
        plt.show()

In [ ]:
def train_model(model, optimizer, data_loader, loss_module, num_epochs=100, plotter=None):
    # Set model to train mode
    model.train()

    if plotter is not None:
        plotter.on_train_begin()


    # Training loop
    for epoch in tqdm(range(num_epochs)):
        epoch_loss = 0.0
        num_batches = 0
        for data_inputs, data_labels in data_loader:

            ## Step 2: Run the model on the input data
          preds = model(data_inputs)
          preds = preds.squeeze(dim=1) # Output is [Batch size, 1], but we want [Batch size]

          ## Step 3: Calculate the loss
          loss = loss_module(preds, data_labels.float())

          ## Step 4: Perform backpropagation
          # Before calculating the gradients, we need to ensure that they are all zero.
          # The gradients would not be overwritten, but actually added to the existing ones.
          optimizer.zero_grad()
          # Perform backpropagation
          loss.backward()
          torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

          ## Step 5: Update the parameters
          optimizer.step()

          epoch_loss += loss.item()
          num_batches += 1

          avg_loss = epoch_loss / num_batches


        if plotter is not None:
            plotter.on_epoch_end(epoch, train_loss=avg_loss)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


def runNN(df, epochs=100, plotter=None):
  train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)


  feature_cols = df.columns.tolist()
  feature_cols.remove("Win")

  scaler = StandardScaler()
  train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
  test_df[feature_cols] = scaler.transform(test_df[feature_cols])



  train_dataset = UFCDataset(train_df, feature_cols=feature_cols, label_col="Win")
  train_data_loader = data.DataLoader(train_dataset, batch_size=16, shuffle=True)

  test_dataset = UFCDataset(test_df, feature_cols=feature_cols, label_col="Win")
  test_data_loader = data.DataLoader(test_dataset, batch_size=16, shuffle=False)


  num_inputs = len(feature_cols)
  num_hidden = 2 * num_inputs

  model = UFCNN(num_inputs=num_inputs, num_hidden=num_hidden, num_outputs=1)


  loss_module = nn.BCEWithLogitsLoss()
  optimizer = torch.optim.SGD(model.parameters(), lr=0.001)
  if plotter is not None:
    train_model(model, optimizer, train_data_loader, loss_module, epochs, plotter)
  else:
    train_model(model, optimizer, train_data_loader, loss_module, epochs)
  state_dict = model.state_dict()
  print(state_dict)
  torch.save(state_dict, "LFKO.tar")

  print("Training Results")
  baseline_train_accuracy, baseline_train_precision, baseline_train_recall = eval_model(model, train_data_loader)
  print("\n\n")
  print("Test Results")
  baseline_test_accuracy, baseline_test_precision, baseline_test_recall = eval_model(model, test_data_loader)

  return model, baseline_train_accuracy, baseline_train_precision, baseline_train_recall, train_data_loader, test_data_loader

Now we have a different problem that we have not had before. The data is overfitting a lot, we have a 75% accuracy on training data but only a 58% accuracy on test data. We are

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
plotter = LiveLossPlot()
model, baseline_train_accuracy, baseline_train_precision, baseline_train_recall, train_data_loader, test_data_loader = runNN(cummulative_df, 100, plotter)

This was out problem. 100 epochs was not enough epocs there is still .6 loss and it has not converged yet. Lets look at it again for 300 and see if it is doing any better.

In [ ]:
model, baseline_train_accuracy, baseline_train_precision, baseline_train_recall, train_data_loader, test_data_loader = runNN(cummulative_df, 300, plotter)

This is much better and seems to be converging close to 300. It still doesn't seem to be convering all the way though. I ran it for 800 epochs and it continued going down as well. After waiting for the 800 epochs to converge I realized that I should use a larger learning rate as to help it converge faster, so we are now trying the learning rate of .01 to see if it converges any faster/if it becomes more accurate.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-4)

model, baseline_train_accuracy, baseline_train_precision, baseline_train_recall, train_data_loader, test_data_loader = runNN(cummulative_df, 300, plotter)

This did converge faster and produce a better result. I then increased the epochs to 500 and let it converge.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-4)

model, baseline_train_accuracy, baseline_train_precision, baseline_train_recall, train_data_loader, test_data_loader = runNN(cummulative_df, 500, plotter)

It again improved. At this point I decided to try increasing the learning rate again to see if it would converge faster than it is right now.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.1, weight_decay=1e-4)

model, baseline_train_accuracy, baseline_train_precision, baseline_train_recall, train_data_loader, test_data_loader = runNN(cummulative_df, 300, plotter)

After letting it run and it still being less than the .01 with less epochs I decided that this is as good as I realistically could do. My theory is that these would converge given enough epochs, but right now increasing the learning rate and the epochs is no longer making much of a difference. I belive that once it converges it will be close to if nor more than logistic regression...but I do not have hours to let these things train.

Only after finishing the report did I realize that I never changed it back to ReLU. So I will now switch it back and run those again...however I am not going to write over the old ones because these take forever to converge and I don't want to have to wait for them all over again and risk that they are worse.

In [ ]:
import torch.nn as nn

class UFCNN(nn.Module):
    def __init__(self, num_inputs, num_hidden, num_outputs):
        super().__init__()

        self.linear1 = nn.Linear(num_inputs, num_hidden)
        self.act_fn = nn.Sigmoid()
        self.linear2 = nn.Linear(num_hidden, num_outputs)

    def forward(self, x):
        # Perform the calculation of the model to determine the prediction
        x = self.linear1(x)
        x = self.act_fn(x)
        x = self.linear2(x)
        return x

In [ ]:
plotter = LiveLossPlot()


In [ ]:
model, baseline_train_accuracy, baseline_train_precision, baseline_train_recall, train_data_loader, test_data_loader = runNN(cummulative_df, 300, plotter)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-4)

model, baseline_train_accuracy, baseline_train_precision, baseline_train_recall, train_data_loader, test_data_loader = runNN(cummulative_df, 300, plotter)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-4)

model, baseline_train_accuracy, baseline_train_precision, baseline_train_recall, train_data_loader, test_data_loader = runNN(cummulative_df, 500, plotter)

This did not help either. So in conclusion, I can make a model that will predict the correct winner 62.75% of the time with a NN and 63% of the time with Logistic Regression. Both of these are above the baseline of always choosing the red fighter (57%) but worse than always choosing favored figher. It is unusual that the NN would be less than the LR model...however I believe that if I had enough time to let the model converge, it would eventually catch up to the LR model.

